# CUDA Enfoque con memoria compartida

Sube `paqueteCudaColab.zip`, ejecuta estas secciones con runtime GPU y descarga `resultadosCuda.zip`.


## 1. Descomprimir paquete

Esta seccion limpia archivos anteriores, descomprime el ZIP generado por el pipeline local y muestra una muestra de entradas RAW y metricas.

In [1]:
!rm -rf salidas metricas resultadosCuda.zip
!unzip -q paqueteCudaColab.zip
!find salidas/rawCuda -type f | head
!head -5 metricas/metricasPrepararCuda.csv

salidas/rawCuda/Bercak Daun/imagen_923_raw.bin
salidas/rawCuda/Bercak Daun/imagen_609_raw.bin
salidas/rawCuda/Bercak Daun/imagen_671_raw.bin
salidas/rawCuda/Bercak Daun/imagen_656_raw.bin
salidas/rawCuda/Bercak Daun/imagen_68_raw.bin
salidas/rawCuda/Bercak Daun/imagen_846_raw.bin
salidas/rawCuda/Bercak Daun/imagen_844_raw.bin
salidas/rawCuda/Bercak Daun/imagen_964_raw.bin
salidas/rawCuda/Bercak Daun/imagen_321_raw.bin
salidas/rawCuda/Bercak Daun/imagen_56_raw.bin
idImagen,clase,rutaRaw,alto,ancho,tipoDato,cantidadElementos,tamanoBytes,estado,error,tiempoPreparacionGlobalSegundos
0,Bercak Daun,salidas/rawCuda/Bercak Daun/imagen_0_raw.bin,512,512,uint8,262144,262144,ok,,199.2215750000032
1,Bercak Daun,salidas/rawCuda/Bercak Daun/imagen_1_raw.bin,512,512,uint8,262144,262144,ok,,199.2215750000032
2,Bercak Daun,salidas/rawCuda/Bercak Daun/imagen_2_raw.bin,512,512,uint8,262144,262144,ok,,199.2215750000032
3,Bercak Daun,salidas/rawCuda/Bercak Daun/imagen_3_raw.bin,512,512,uint8,262144,262144,

## 2. Entorno CUDA y kernel

Esta seccion verifica la GPU, carga `nvcc4jupyter` y ejecuta el kernel personalizado. Cada imagen se procesa como una matriz `512x512` y se aplica el filtro de enfoque:

```text
[  0  -1   0 ]
[ -1   5  -1 ]
[  0  -1   0 ]
```


In [2]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-pmtnbazq
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-pmtnbazq
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10741 sha256=90a829d54142cdbaf5703d6409de30ec0002408286cfbcd108dfdd84cf0ee4fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-m3mg3ipj/wheels/7d/b9/66/459b9938664e6a93d1a85323ec52f7e51cd7265d253410a7d8
Successfully built nvcc4jupyter


In [3]:
!pip install nvcc4jupyter

In [4]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpdgy0hrpr".


In [5]:
!nvidia-smi

Sun May 17 00:22:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [7]:
%load_ext nvcc4jupyter
%reload_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpyijftosg".


### **Aplicación CUDA**

In [8]:
%%cuda
#include <cuda_runtime.h>

#include <algorithm>
#include <chrono>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <sstream>
#include <string>
#include <vector>

using namespace std;

// ###### CONFIGURACION CUDA ######
const int HILOS_X = 16;
const int HILOS_Y = 16;

// ###### ESTRUCTURA DE CADA IMAGEN ######
struct ImagenEntrada {
    int idImagen;
    string clase;
    string rutaRaw;
    int alto;
    int ancho;
};

// ###### KERNEL GPU: FILTRO DE ENFOQUE CON MEMORIA COMPARTIDA ######
// Cada imagen llega como una matriz 512x512 uint8 preparada con NumPy/OpenCV.
// El kernel aplica esta matriz de convolucion de enfoque sobre cada pixel:
//
//        [  0  -1   0 ]
//   E =  [ -1   5  -1 ]
//        [  0  -1   0 ]
//
// La memoria compartida guarda un tile del bloque mas un borde de 1 pixel
// para reutilizar los vecinos y reducir lecturas repetidas desde memoria global.
__global__ void aplicarEnfoque(unsigned char* entrada, unsigned char* salida, int alto, int ancho) {
    __shared__ unsigned char tile[HILOS_Y + 2][HILOS_X + 2];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int x = blockIdx.x * blockDim.x + tx;
    int y = blockIdx.y * blockDim.y + ty;
    int lx = tx + 1;
    int ly = ty + 1;

    if (x < ancho && y < alto) {
        tile[ly][lx] = entrada[y * ancho + x];
    } else {
        tile[ly][lx] = 0;
    }

    if (tx == 0) {
        tile[ly][0] = (x > 0 && y < alto) ? entrada[y * ancho + (x - 1)] : 0;
    }
    if (tx == blockDim.x - 1) {
        tile[ly][HILOS_X + 1] = (x + 1 < ancho && y < alto) ? entrada[y * ancho + (x + 1)] : 0;
    }
    if (ty == 0) {
        tile[0][lx] = (y > 0 && x < ancho) ? entrada[(y - 1) * ancho + x] : 0;
    }
    if (ty == blockDim.y - 1) {
        tile[HILOS_Y + 1][lx] = (y + 1 < alto && x < ancho) ? entrada[(y + 1) * ancho + x] : 0;
    }
    if (tx == 0 && ty == 0) {
        tile[0][0] = (x > 0 && y > 0) ? entrada[(y - 1) * ancho + (x - 1)] : 0;
    }
    if (tx == blockDim.x - 1 && ty == 0) {
        tile[0][HILOS_X + 1] = (x + 1 < ancho && y > 0) ? entrada[(y - 1) * ancho + (x + 1)] : 0;
    }
    if (tx == 0 && ty == blockDim.y - 1) {
        tile[HILOS_Y + 1][0] = (x > 0 && y + 1 < alto) ? entrada[(y + 1) * ancho + (x - 1)] : 0;
    }
    if (tx == blockDim.x - 1 && ty == blockDim.y - 1) {
        tile[HILOS_Y + 1][HILOS_X + 1] = (x + 1 < ancho && y + 1 < alto) ? entrada[(y + 1) * ancho + (x + 1)] : 0;
    }

    __syncthreads();

    if (x >= ancho || y >= alto) {
        return;
    }

    int idx = y * ancho + x;
    if (x == 0 || y == 0 || x == ancho - 1 || y == alto - 1) {
        salida[idx] = 0;
        return;
    }

    int valor =
        -1 * tile[ly - 1][lx]
        -1 * tile[ly][lx - 1]
        +5 * tile[ly][lx]
        -1 * tile[ly][lx + 1]
        -1 * tile[ly + 1][lx];
    salida[idx] = (unsigned char)min(max(valor, 0), 255);
}

// ###### IO: LEER CSV SIMPLE GENERADO POR PYTHON ######
vector<string> separarCsv(const string& linea) {
    vector<string> columnas;
    string actual;
    for (char c : linea) {
        if (c == ',') {
            columnas.push_back(actual);
            actual.clear();
        } else {
            actual += c;
        }
    }
    columnas.push_back(actual);
    return columnas;
}

string normalizarRuta(string ruta) {
    replace(ruta.begin(), ruta.end(), '\\', '/');
    return ruta;
}

vector<ImagenEntrada> leerImagenesEntrada() {
    ifstream archivo("metricas/metricasPrepararCuda.csv");
    if (!archivo.is_open()) {
        throw runtime_error("No se pudo abrir metricas/metricasPrepararCuda.csv");
    }

    vector<ImagenEntrada> imagenes;
    string linea;

    getline(archivo, linea);
    while (getline(archivo, linea)) {
        vector<string> c = separarCsv(linea);
        if (c.size() >= 9 && c[8] == "ok") {
            imagenes.push_back({stoi(c[0]), c[1], normalizarRuta(c[2]), stoi(c[3]), stoi(c[4])});
        }
    }
    return imagenes;
}

// ###### IO: ARCHIVOS BINARIOS RAW ######
void crearCarpetaSalida(const string& rutaArchivo) {
    size_t pos = rutaArchivo.find_last_of('/');
    if (pos != string::npos) {
        string comando = "mkdir -p \"" + rutaArchivo.substr(0, pos) + "\"";
        system(comando.c_str());
    }
}

void leerRaw(const string& ruta, unsigned char* datos, size_t bytes) {
    ifstream archivo(ruta, ios::binary);
    if (!archivo.is_open()) {
        throw runtime_error("No se pudo abrir RAW: " + ruta);
    }

    archivo.read((char*)datos, bytes);
    if ((size_t)archivo.gcount() != bytes) {
        throw runtime_error("RAW incompleto: " + ruta);
    }
}

void guardarRaw(const string& ruta, unsigned char* datos, size_t bytes) {
    crearCarpetaSalida(ruta);
    ofstream archivo(ruta, ios::binary);
    if (!archivo.is_open()) {
        throw runtime_error("No se pudo guardar CUDA BIN: " + ruta);
    }
    archivo.write((char*)datos, bytes);
}

// ###### GPU: RESERVAR MEMORIA, COPIAR, LANZAR KERNEL Y RETORNAR ######
float procesarEnGpu(unsigned char* hEntrada, unsigned char* hSalida, int alto, int ancho) {
    size_t bytes = alto * ancho * sizeof(unsigned char);

    unsigned char* dEntrada;
    unsigned char* dSalida;
    cudaMalloc(&dEntrada, bytes);
    cudaMalloc(&dSalida, bytes);

    cudaMemcpy(dEntrada, hEntrada, bytes, cudaMemcpyHostToDevice);

    dim3 hilos(HILOS_X, HILOS_Y);
    dim3 bloques((ancho + hilos.x - 1) / hilos.x, (alto + hilos.y - 1) / hilos.y);

    cudaEvent_t inicio, fin;
    cudaEventCreate(&inicio);
    cudaEventCreate(&fin);
    cudaEventRecord(inicio);

    aplicarEnfoque<<<bloques, hilos>>>(dEntrada, dSalida, alto, ancho);
    cudaDeviceSynchronize();

    cudaEventRecord(fin);
    cudaEventSynchronize(fin);

    float tiempoKernelMs = 0.0f;
    cudaEventElapsedTime(&tiempoKernelMs, inicio, fin);

    cudaMemcpy(hSalida, dSalida, bytes, cudaMemcpyDeviceToHost);

    cudaFree(dEntrada);
    cudaFree(dSalida);
    cudaEventDestroy(inicio);
    cudaEventDestroy(fin);

    return tiempoKernelMs;
}

// ###### MAIN: PROCESA TODAS LAS IMAGENES DEL PAQUETE ######
int main() {
    system("mkdir -p metricas salidas/cudaBin");

    vector<ImagenEntrada> imagenes = leerImagenesEntrada();
    ofstream metricas("metricas/metricasCudaColab.csv");
    metricas << "idImagen,clase,rutaRaw,rutaCudaBin,alto,ancho,tiempoKernelMs,tiempoTotalGpuSegundos,hilosX,hilosY,filtro,estado,error\n";

    for (ImagenEntrada img : imagenes) {
        auto inicioTotal = chrono::high_resolution_clock::now();
        string rutaSalida = "salidas/cudaBin/" + img.clase + "/imagen_" + to_string(img.idImagen) + "_cuda.bin";
        size_t bytes = img.alto * img.ancho * sizeof(unsigned char);

        unsigned char* hEntrada = (unsigned char*)malloc(bytes);
        unsigned char* hSalida = (unsigned char*)malloc(bytes);

        try {
            leerRaw(img.rutaRaw, hEntrada, bytes);
            float tiempoKernelMs = procesarEnGpu(hEntrada, hSalida, img.alto, img.ancho);
            guardarRaw(rutaSalida, hSalida, bytes);

            auto finTotal = chrono::high_resolution_clock::now();
            chrono::duration<double> tiempoTotal = finTotal - inicioTotal;

            metricas << img.idImagen << "," << img.clase << "," << img.rutaRaw << "," << rutaSalida
                     << "," << img.alto << "," << img.ancho << "," << tiempoKernelMs << ","
                     << tiempoTotal.count() << "," << HILOS_X << "," << HILOS_Y << ",enfoque,ok,\n";
        } catch (...) {
            auto finTotal = chrono::high_resolution_clock::now();
            chrono::duration<double> tiempoTotal = finTotal - inicioTotal;

            metricas << img.idImagen << "," << img.clase << "," << img.rutaRaw << "," << rutaSalida
                     << "," << img.alto << "," << img.ancho << ",0," << tiempoTotal.count()
                     << "," << HILOS_X << "," << HILOS_Y << ",enfoque,error,error en CUDA o lectura\n";
        }

        free(hEntrada);
        free(hSalida);
    }

    cout << "Procesamiento CUDA Enfoque completado." << endl;
    return 0;
}


Procesamiento CUDA Enfoque completado.



## 3. Descargar resultados

Esta seccion empaqueta los binarios generados por CUDA y las metricas para integrarlos de vuelta en el proyecto local.

In [9]:
!zip -r resultadosCuda.zip salidas/cudaBin metricas/metricasCudaColab.csv
from google.colab import files
files.download("resultadosCuda.zip")

  adding: salidas/cudaBin/ (stored 0%)
  adding: salidas/cudaBin/Bercak Daun/ (stored 0%)
  adding: salidas/cudaBin/Bercak Daun/imagen_819_cuda.bin (deflated 5%)
  adding: salidas/cudaBin/Bercak Daun/imagen_497_cuda.bin (deflated 12%)
  adding: salidas/cudaBin/Bercak Daun/imagen_416_cuda.bin (deflated 17%)
  adding: salidas/cudaBin/Bercak Daun/imagen_377_cuda.bin (deflated 10%)
  adding: salidas/cudaBin/Bercak Daun/imagen_153_cuda.bin (deflated 9%)
  adding: salidas/cudaBin/Bercak Daun/imagen_83_cuda.bin (deflated 6%)
  adding: salidas/cudaBin/Bercak Daun/imagen_763_cuda.bin (deflated 11%)
  adding: salidas/cudaBin/Bercak Daun/imagen_664_cuda.bin (deflated 10%)
  adding: salidas/cudaBin/Bercak Daun/imagen_24_cuda.bin (deflated 4%)
  adding: salidas/cudaBin/Bercak Daun/imagen_715_cuda.bin (deflated 13%)
  adding: salidas/cudaBin/Bercak Daun/imagen_320_cuda.bin (deflated 14%)
  adding: salidas/cudaBin/Bercak Daun/imagen_398_cuda.bin (deflated 27%)
  adding: salidas/cudaBin/Bercak Daun/im

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>